# Thermo Fisher Cytomat

The Cytomat series of incubators stores microplates under controlled environmental conditions. PLR supports several models through two backends:

- {class}`~pylabrobot.thermo_fisher.cytomat.backend.CytomatBackend` -- for modern Cytomat models (serial protocol)
- {class}`~pylabrobot.thermo_fisher.cytomat.heraeus_backend.HeraeusCytomatBackend` -- for legacy Heraeus-era Cytomats (PLC protocol)

| Model | `CytomatType` value | Shaking |
|---|---|---|
| C6000 | `CytomatType.C6000` | yes |
| C6002 | `CytomatType.C6002` | yes |
| C2C 425 | `CytomatType.C2C_425` | yes |
| C2C 450 Shake | `CytomatType.C2C_450_SHAKE` | yes |
| C5C | `CytomatType.C5C` | no |

Capabilities:

- [Automated retrieval](../../capabilities/automated-retrieval) -- store and fetch plates by slot
- [Temperature control](../../capabilities/temperature-control) -- read incubation temperature (set via device UI)
- [Humidity control](../../capabilities/humidity-control) -- read humidity (set via device UI)
- [Shaking](../../capabilities/shaking) -- integrated shakers (all models except C5C)

A {class}`~pylabrobot.thermo_fisher.cytomat.chatterbox.CytomatChatterbox` backend is available for testing without hardware.

## Setup

Create a {class}`~pylabrobot.thermo_fisher.cytomat.backend.CytomatBackend` with the model type and serial port, configure racks, and build the {class}`~pylabrobot.thermo_fisher.cytomat.cytomat.Cytomat` resource.

In [ ]:
from pylabrobot.thermo_fisher.cytomat import Cytomat, CytomatBackend, CytomatType
from pylabrobot.thermo_fisher.cytomat.racks import cytomat_rack_9mm_51
from pylabrobot.resources import Coordinate

backend = CytomatBackend(model=CytomatType.C6000, port="/dev/ttyUSB0")  # replace with your port

rack = cytomat_rack_9mm_51("rack_A")
cytomat = Cytomat(
    name="cytomat",
    driver=backend,
    racks=[rack],
    loading_tray_location=Coordinate(0, 0, 0),
    size_x=860,
    size_y=550,
    size_z=900,
)

await cytomat.setup()

Many rack configurations are available in `pylabrobot.thermo_fisher.cytomat.racks` -- for example `cytomat_rack_10mm_47`, `cytomat_rack_17mm_28`, `cytomat_rack_23mm_21`, etc. Pick the one that matches your physical rack pitch and slot count.

## Storing a plate

Place a plate on the loading tray and call {meth}`~pylabrobot.thermo_fisher.cytomat.cytomat.Cytomat.take_in_plate` to move it into storage. You can specify `"smallest"` (smallest free site that fits), `"random"`, or an explicit site. See [Automated Retrieval](../../capabilities/automated-retrieval) for the full capability API.

In [ ]:
from pylabrobot.resources.corning.plates import Cor_96_wellplate_360ul_Fb

plate = Cor_96_wellplate_360ul_Fb("my_plate")
cytomat.loading_tray.assign_child_resource(plate)

await cytomat.take_in_plate("smallest")  # choose the smallest free site

# other options:
# await cytomat.take_in_plate("random")       # random free site
# await cytomat.take_in_plate(rack[3])         # store at rack position 3

## Retrieving a plate

Use {meth}`~pylabrobot.thermo_fisher.cytomat.cytomat.Cytomat.fetch_plate_to_loading_tray` to move a plate from storage back to the loading tray.

In [ ]:
await cytomat.fetch_plate_to_loading_tray("my_plate")
retrieved = cytomat.loading_tray.resource

## Temperature and humidity monitoring

The Cytomat exposes read-only {class}`~pylabrobot.capabilities.temperature_controlling.temperature_controller.TemperatureController` and {class}`~pylabrobot.capabilities.humidity_controlling.humidity_controller.HumidityController` capabilities. Temperature and humidity set-points are configured on the device UI -- PLR can only query current values. See [Temperature Control](../../capabilities/temperature-control) and [Humidity Control](../../capabilities/humidity-control) for the full capability APIs.

In [ ]:
current_temp = await cytomat.tc.request_temperature()
current_humidity = await cytomat.humidity.request_humidity()
print(f"Temperature: {current_temp:.1f} C, Humidity: {current_humidity:.1f} %")

## Shaking

All models except C5C have integrated shakers exposed as a {class}`~pylabrobot.capabilities.shaking.shaking.Shaker`. See [Shaking](../../capabilities/shaking) for the full capability API.

In [ ]:
# Start shaking at 500 rpm
await cytomat.shaker.shake(speed=500)

# Stop shaking
await cytomat.shaker.stop_shaking()

## Low-level backend operations

The backend exposes additional device-specific commands for door control, swap station queries, and direct storage-to-transfer movements:

In [ ]:
# Door control
await backend.open_door()
await backend.close_door()

# Query incubation conditions (returns nominal and actual values)
co2 = await backend.request_co2()
o2 = await backend.request_o2()
print(f"CO2: nominal={co2.nominal_value}, actual={co2.actual_value}")
print(f"O2:  nominal={o2.nominal_value}, actual={o2.actual_value}")

# Wait for the transfer station to become occupied
await backend.wait_for_transfer_station(occupied=True)

## Storage summary

Call `summary()` on the Cytomat to get a table showing which slots are occupied:

In [ ]:
print(cytomat.summary())

## Legacy Heraeus backend

For older Heraeus-era Cytomats, use {class}`~pylabrobot.thermo_fisher.cytomat.heraeus_backend.HeraeusCytomatBackend` instead. It speaks a PLC-based serial protocol but exposes the same capabilities:

In [ ]:
from pylabrobot.thermo_fisher.cytomat import HeraeusCytomatBackend

heraeus_backend = HeraeusCytomatBackend(port="/dev/ttyUSB1")  # replace with your port

cytomat_legacy = Cytomat(
    name="cytomat_legacy",
    driver=heraeus_backend,
    racks=[cytomat_rack_9mm_51("rack_B")],
    loading_tray_location=Coordinate(0, 0, 0),
    size_x=860,
    size_y=550,
    size_z=900,
)

## Teardown

In [ ]:
await cytomat.stop()